In [ ]:
# ── The unattended-run contract ───────────────────────────────────────────────
# This notebook is launched by `.github/workflows/model_retrain.yml` on a Colab T4,
# followed every twenty minutes by `model_retrain_watch.yml`, and collected by
# `model_intake.yml`. None of them can see this VM, so everything they need to know is
# written to files with a schema — by `scripts/mlops/retrain_state.py`, the same module
# those workflows import. One definition, two machines.
#
# Run it by hand and it still works: with no `run_config.json` on the VM, the defaults
# below describe an interactive run and nothing is published anywhere.
import json, os, shutil, subprocess, sys, time
from pathlib import Path

RUN_CONFIG_PATH = os.environ.get('HN_RUN_CONFIG', '/content/hn_retrain/run_config.json')
REPO_DIR_LOCAL = Path(os.environ.get('HN_REPO_DIR', '/content/hn_retrain/repo'))

if not (REPO_DIR_LOCAL / 'scripts' / 'mlops' / 'retrain_state.py').is_file():
    REPO_DIR_LOCAL.parent.mkdir(parents=True, exist_ok=True)
    # Anonymous and shallow: the notebook needs the contract module and nothing else. A
    # public clone needs no credential, which is the point — this notebook used to prompt
    # for a fine-grained PAT with `getpass`, and a prompt nobody can answer hangs an
    # unattended run until the session is recycled.
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/myself-aas/HazardNet.git', str(REPO_DIR_LOCAL)],
        check=True,
    )
sys.path.insert(0, str(REPO_DIR_LOCAL / 'scripts'))
from mlops import retrain_state as rs  # noqa: E402

RUN_CONFIG = rs.read_json(RUN_CONFIG_PATH) or {}
UNATTENDED = bool(RUN_CONFIG.get('unattended'))
RUN_STRATEGY = RUN_CONFIG.get('strategy') or 'event_kfold'
RUN_ID = RUN_CONFIG.get('run_id') or rs.new_run_id(RUN_STRATEGY)
RESUME = bool(RUN_CONFIG.get('resume', UNATTENDED))
VM_RUN_DIR = Path(RUN_CONFIG.get('vm_run_dir') or f'{rs.VM_RUN_ROOT}/{RUN_ID}')
# Where fold checkpoints go. Actions says so explicitly, because a headless session has no
# Drive and the answer therefore differs from an interactive one; the Drive cell below
# resolves it once the mount situation is known.
RUN_CHECKPOINT_DIR = RUN_CONFIG.get('checkpoint_dir')
DRIVE_RUN_DIR = Path(RUN_CONFIG.get('drive_run_dir') or f'{rs.DRIVE_ROOT}/{rs.DRIVE_RETRAIN_SUBDIR}/{RUN_ID}')
NOTEBOOK_SHA256 = RUN_CONFIG.get('notebook_sha256', '')
REPO_SHA = RUN_CONFIG.get('repo_sha', '')
RUN_STARTED_AT = time.time()
RUN_STARTED_STAMP = rs.stamp()
RESUMED_FROM = []


def beat(phase, **fields):
    """Tell the watcher this session is alive.

    Cheap, and the only signal that crosses from this VM to a GitHub runner. The watcher
    reads it to tell "inside a long epoch" from "the session was recycled forty minutes
    ago" — and those two need opposite responses: leave it alone, or relaunch with
    `resume: true` and let the checkpoints carry on.
    """
    VM_RUN_DIR.mkdir(parents=True, exist_ok=True)
    doc = rs.heartbeat_doc(RUN_ID, phase=phase, **fields)
    rs.write_heartbeat(VM_RUN_DIR, doc, mirror_dir=DRIVE_RUN_DIR if DRIVE_RUN_DIR.exists() else None)
    return doc


RESUMED_FROM.extend(path.name for path in rs.resume_states(DRIVE_RUN_DIR))
beat('starting', message=('resuming' if RESUMED_FROM else 'fresh') + f' run under {RUN_STRATEGY}')

print(f'run {RUN_ID}  strategy={RUN_STRATEGY}  unattended={UNATTENDED}  resume={RESUME}')
print(f'  vm     {VM_RUN_DIR}')
print(f'  drive  {DRIVE_RUN_DIR}')
print(f'  parity gate {rs.PARITY_GATE_PCT}% hazard agreement (PyTorch vs TFLite)')
if RESUMED_FROM:
    print(f'  ↻ resumable states left by a dead session: {", ".join(RESUMED_FROM)}')

In [ ]:
# Drive is optional, and mounting it is interactive: `drive.mount()` prompts for a code and
# the CLI's `colab drivemount` needs a TTY and browser consent. A headless session — which
# is every session Actions launches — therefore has no Drive at all, and nothing downstream
# may assume one. Mount only when it is already there or a human is driving.
DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
if not DRIVE_MOUNTED and not UNATTENDED:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
print(f'Drive mounted: {DRIVE_MOUNTED}'
      + ('' if DRIVE_MOUNTED else '  (headless run — everything falls back to VM-local paths)'))

In [ ]:
# One spelling for the Drive root, taken from the contract module.
#
# This cell used to `%cd` into "HazardNet Deployment" (with a space) while every later cell
# read "HazardNet_Deployment" (with an underscore), so the working directory and the data
# paths disagreed. Interactively that is an inconvenience; at hour six of an unattended run
# it is a failure nobody is awake to see.
DRIVE_ROOT = Path(rs.DRIVE_ROOT)
if DRIVE_MOUNTED:
    RUN_DRIVE_DIR = DRIVE_RUN_DIR
    RUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / 'data').mkdir(exist_ok=True)
    os.chdir(DRIVE_ROOT)
else:
    RUN_DRIVE_DIR = None

# Checkpoints: Drive when a human mounted it (then they survive a recycled session on their
# own), the VM otherwise — in which case the watcher carries the newest one out to a
# workflow artifact every tick and uploads it back into any replacement VM.
CHECKPOINT_DIR = Path(RUN_CHECKPOINT_DIR or rs.checkpoint_dir(str(VM_RUN_DIR), DRIVE_MOUNTED))
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HN_OUTPUT_DIR'] = str(CHECKPOINT_DIR)

beat('mounted', message=f'drive={DRIVE_MOUNTED} checkpoints={CHECKPOINT_DIR}')
print(f'working directory: {os.getcwd()}')
print(f'checkpoints:       {CHECKPOINT_DIR}')
print(f'run directory:     {RUN_DRIVE_DIR or VM_RUN_DIR}')

In [ ]:
# Where the master tensor comes from, in the order that survives the most situations.
#
# A headless session has no Drive, so an unattended run cannot rely on the tensor being
# mounted; it downloads the same Kaggle dataset the interactive path was built from, using
# credentials Actions uploads to the VM. Interactively, Drive stays the fast path: the first
# download happens once and every later run is instant.
TENSOR_NAME = 'master_tensors.h5'
DRIVE_TENSOR = DRIVE_ROOT / 'tensors_output' / 'HazardNet_Event_Based_Datasets' / TENSOR_NAME
LOCAL_DATA = Path('/content/data')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

configured = RUN_CONFIG.get('tensor_path')
if configured and Path(configured).is_file():
    TENSOR_DIR = Path(configured).parent
    print(f'· tensor from run_config: {configured}')
elif DRIVE_MOUNTED and DRIVE_TENSOR.is_file():
    TENSOR_DIR = DRIVE_TENSOR.parent
    print(f'· tensor from Drive ({DRIVE_TENSOR.stat().st_size / 1e9:.2f} GB)')
else:
    KAGGLE_DIR = LOCAL_DATA / 'kaggle'
    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    assert (Path.home() / '.kaggle' / 'kaggle.json').is_file(), (
        f'no tensor at {configured or DRIVE_TENSOR} and no Kaggle credentials on this VM. An '
        'unattended run needs KAGGLE_USERNAME/KAGGLE_KEY uploaded as ~/.kaggle/kaggle.json; an '
        'interactive one needs Drive mounted with the tensor in it.'
    )
    print('· downloading the dataset from Kaggle (a headless session has no Drive)')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'ashifahmedshuvo/hazardnet-datasets',
                    '-p', str(KAGGLE_DIR), '--unzip'], check=True)
    hits = sorted(KAGGLE_DIR.rglob(TENSOR_NAME))
    assert hits, (
        f'the Kaggle dataset did not contain {TENSOR_NAME}; it holds '
        f'{[str(p.relative_to(KAGGLE_DIR)) for p in sorted(KAGGLE_DIR.rglob("*")) if p.is_file()][:20]}'
    )
    TENSOR_DIR = hits[0].parent
    print(f'· tensor from Kaggle: {hits[0]} ({hits[0].stat().st_size / 1e9:.2f} GB)')

TENSOR_SRC = TENSOR_DIR / TENSOR_NAME
assert TENSOR_SRC.is_file(), f'{TENSOR_SRC} is not a file'
# The fold CSVs and dataset_config.json sit beside the tensor, and the converter reads them
# from there — so this directory, not a Drive path, is what training is told about.
os.environ['HN_TENSOR_DIR'] = str(TENSOR_DIR)
beat('dataset', message=str(TENSOR_SRC))

In [ ]:
# Configuration. No credentials, and nothing in this notebook talks to GitHub any more.
#
# The `getpass` PAT prompt that used to be here is gone for two reasons: an unattended run
# cannot answer a prompt (it hangs until the session is recycled, and the watcher then
# relaunches a run that was never going to start), and a fine-grained token held in a
# notebook's memory is a standing write credential to the repository. Every git operation
# this pipeline needs now happens in Actions, with `GITHUB_TOKEN`, scoped to one run.
DATASET_SOURCE = "drive"
DRIVE_TENSORS_PATH = str(TENSOR_SRC)
BUNDLE_DIR = str(DRIVE_ROOT / 'HazardNet_Deployment_Bundles' / 'deployment_bundle')
REPO_DIR = str(REPO_DIR_LOCAL)
print("Configured.")
print(f"  tensors {DRIVE_TENSORS_PATH}")
print(f"  bundle  {BUNDLE_DIR}")

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — Runtime → Change runtime type → GPU"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | Device {torch.cuda.get_device_name(0)}")

In [ ]:
# Copy the tensor onto the VM's own disk and point training at the copy.
#
# The old version copied it from Drive to another Drive folder, so every batch of every
# epoch was read back through the FUSE mount — the slowest way to feed a GPU, and slow
# epochs are what make a live run look dead to anything watching it from outside.
LOCAL_DATA_DIR = Path('/content/data')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
TENSORS_PATH = str(LOCAL_DATA_DIR / 'master_tensors.h5')

if str(TENSOR_SRC).startswith('/content/'):
    # Already on the VM's own disk (a Kaggle download, or a path Actions staged): copying
    # it again would only fill the disk twice.
    TENSORS_PATH = str(TENSOR_SRC)
elif DATASET_SOURCE == 'drive':
    # A size check rather than an existence check: a copy interrupted by a recycled session
    # leaves a truncated file behind, and training on one fails in a way that looks like a
    # data bug.
    if not (os.path.exists(TENSORS_PATH) and os.path.getsize(TENSORS_PATH) == TENSOR_SRC.stat().st_size):
        shutil.copy(str(TENSOR_SRC), TENSORS_PATH)
elif DATASET_SOURCE == 'hf':
    # This branch used to interpolate an `HF_DATASET_REPO` that nothing ever defined, so
    # it could only ever fail. Say so instead of guessing at a repository.
    raise SystemExit("DATASET_SOURCE == 'hf' is not wired up; the supported source is 'drive'")
else:
    raise SystemExit(f'unknown DATASET_SOURCE {DATASET_SOURCE!r}')

assert os.path.exists(TENSORS_PATH), f"Dataset not found at {TENSORS_PATH}"
# `TrainConfig` reads this environment variable: it is the only reason the local copy is
# used at all, so setting it here is what makes the copy matter.
os.environ['HN_MASTER_H5'] = TENSORS_PATH
beat('tensor-local', message=f'{os.path.getsize(TENSORS_PATH) / 1e9:.2f} GB at {TENSORS_PATH}')
print(f'{os.path.getsize(TENSORS_PATH) / 1e9:.2f} GB local at {TENSORS_PATH}')

In [ ]:
# Training stack (only needed in Colab — the Actions daily job uses tflite-runtime only).
!pip install -q torch torchvision h5py numpy pandas scikit-learn tqdm onnxruntime onnx tf2onnx tensorflow tensorflow-probability
import torch, torch.nn as nn, numpy as np, h5py, pandas as pd
from tqdm import tqdm
print("Training deps installed.")

In [ ]:
"""
================================================================================
HazardNet Unified Experimental Training Pipeline (Q1 Journal Edition)
================================================================================

INTEGRATED FEATURES:
  • W&B Kaggle Secrets Integration (robust fallback)
  • Enhanced Publication Metrics (Per-class, Severity Quartiles, R²)
  • Publication Figure Generator (Confusion Matrix, Scatter, Spatial Heatmap)
  • 4 Validation Strategies: Event K-Fold, Spatial LODO, Temporal, Spatio-Temporal

USAGE IN KAGGLE NOTEBOOK CELL:
  STRATEGY = 'spatial_lodo'  # Change per session
  main()
================================================================================
"""

import os
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, get_worker_info
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             mean_squared_error, mean_absolute_error, confusion_matrix,
                             r2_score)
from tqdm import tqdm
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# W&B is off, and the import has to be conditional along with it: the training loop calls
# `wandb.log` unguarded, so flipping this flag without the package installed used to be a
# NameError at the end of the first epoch — hours of GPU lost to a logging preference.
WANDB_ENABLED = False
try:
    import wandb
except ImportError:
    wandb = None
if wandb is None:
    WANDB_ENABLED = False

# ============================================================================
# CONFIGURATION
# ============================================================================
class TrainConfig:
    # The directory the dataset cell resolved: Drive when it is mounted, the Kaggle download
    # when it is not. The fold CSVs and dataset_config.json live beside the tensor, so this
    # has to follow the tensor rather than assume a Drive path.
    EXPERIMENTAL_DIR = os.environ.get(
        'HN_TENSOR_DIR', os.path.join(str(DRIVE_ROOT), 'tensors_output', 'HazardNet_Event_Based_Datasets'))
    # The local copy the dataset cell made (HN_MASTER_H5). Reading 5 GB of HDF5 batch by
    # batch through the Drive FUSE mount is the slowest way to feed a GPU, and slow epochs
    # are what make a live run look dead to anything watching from outside.
    MASTER_H5_PATH = os.environ.get('HN_MASTER_H5', os.path.join(EXPERIMENTAL_DIR, 'master_tensors.h5'))
    CONFIG_PATH = os.path.join(EXPERIMENTAL_DIR, 'dataset_config.json')
    # Checkpoints stay on Drive on purpose: a recycled session takes /content with it, and
    # a resume is only possible from a file that outlived the VM.
    OUTPUT_DIR = os.environ.get(
        'HN_OUTPUT_DIR', os.path.join(str(DRIVE_ROOT), 'data', 'HazardNet_event_based_model_outputs'))

    # Set by the run contract cell. RESUME is what model_retrain_watch.yml turns on when it
    # relaunches a run whose session was recycled.
    RESUME = bool(RESUME)
    RUN_ID = str(RUN_ID)
    #: How often a resumable state is written. Every epoch: on a free tier the expected
    #: cost of losing one is an hour of GPU, and the write is tens of megabytes to Drive.
    CHECKPOINT_EVERY = int(os.environ.get('HN_CHECKPOINT_EVERY', '1'))

    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 10
    GRAD_CLIP = 1.0
    NUM_WORKERS = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(TrainConfig.OUTPUT_DIR, exist_ok=True)

HAZARD_TYPES = [
    'Cold Wave', 'Drought', 'Fire', 'Flash Flood',
    'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone'
]

# ============================================================================
# HAZARDNET ARCHITECTURE
# ============================================================================
class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size,
                                   padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return self.bn(self.pointwise(self.depthwise(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(
            DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True),
            SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(
            DepthwiseSeparableConv3d(32, 64), nn.ReLU(True),
            SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(
            DepthwiseSeparableConv3d(64, 128), nn.ReLU(True),
            SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(
            DepthwiseSeparableConv3d(128, 256), nn.ReLU(True),
            SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ============================================================================
# LOSS FUNCTION
# ============================================================================
class HomoscedasticMTLLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(2))
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')
        self.huber_loss = nn.SmoothL1Loss(reduction='none')

    def forward(self, hazard_pred, severity_pred, hazard_true, severity_true, confidence):
        loss_cls = self.ce_loss(hazard_pred, hazard_true)
        loss_reg = self.huber_loss(severity_pred, severity_true)
        loss_cls_conf = (loss_cls * confidence).mean()
        loss_reg_conf = (loss_reg * confidence).mean()
        prec_cls = torch.exp(-self.log_vars[0])
        prec_reg = torch.exp(-self.log_vars[1])
        total = (prec_cls * loss_cls_conf + self.log_vars[0]) + \
                (prec_reg * loss_reg_conf + self.log_vars[1])
        return total, loss_cls_conf.item(), loss_reg_conf.item()


# ============================================================================
# MASTER HDF5 DATASET
# ============================================================================
class MasterHDF5Dataset(Dataset):
    def __init__(self, csv_path, master_h5_path, augment=False):
        self.df = pd.read_csv(csv_path)
        self.master_h5_path = master_h5_path
        self.augment = augment
        self.h5f = None
        self._worker_id = None
        self.brightness, self.contrast, self.temporal_shift = 0.1, 0.1, 1
        self.target_shape = (15, 10, 64, 64)

    def _open_h5(self):
        wid = get_worker_info().id if get_worker_info() else -1
        if self.h5f is None or self._worker_id != wid:
            if self.h5f: self.h5f.close()
            self.h5f = h5py.File(self.master_h5_path, 'r', rdcc_nbytes=1024**2*10)
            self._worker_id = wid

    def __len__(self): return len(self.df)

    def _resize_spatial(self, tensor):
        c, t, h, w = tensor.shape
        th, tw = self.target_shape[2], self.target_shape[3]
        if h == th and w == tw: return tensor
        r = tensor.permute(1,0,2,3).reshape(t*c,1,h,w)
        r = F.interpolate(r, size=(th,tw), mode='nearest')
        return r.reshape(t,c,th,tw).permute(1,0,2,3).contiguous()

    def _augment(self, tensor):
        if np.random.rand() > 0.5:
            tensor = tensor + np.random.uniform(-self.brightness, self.brightness)
        if np.random.rand() > 0.5:
            f = 1.0 + np.random.uniform(-self.contrast, self.contrast)
            m = tensor.mean(dim=[-1,-2], keepdim=True)
            tensor = (tensor - m) * f + m
        if np.random.rand() > 0.5:
            s = np.random.randint(-self.temporal_shift, self.temporal_shift+1)
            if s > 0:
                b = tensor[:,0:1,:,:].repeat(1,s,1,1)
                tensor = torch.cat([b, tensor[:,:-s,:,:]], dim=1)
            elif s < 0:
                a = abs(s); b = tensor[:,-1:,:,:].repeat(1,a,1,1)
                tensor = torch.cat([tensor[:,a:,:,:], b], dim=1)
        return tensor

    def __getitem__(self, idx):
        self._open_h5()
        row = self.df.iloc[idx]
        eid = str(row['event_id'])
        tensor = torch.from_numpy(self.h5f['tensors'][eid][:]).float()
        label = int(row['hazard_idx'])
        severity = float(row.get('severity_index', 0.0))
        confidence = float(row.get('confidence', 0.5))
        tensor = self._resize_spatial(tensor)
        if self.augment: tensor = self._augment(tensor)
        return tensor, label, severity, confidence, eid

    def __del__(self):
        if self.h5f: self.h5f.close()


# ============================================================================
# ENHANCED METRICS TRACKER (Q1 Journal Grade)
# ============================================================================
class EnhancedMetricsTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.total_losses, self.cls_losses, self.reg_losses = [], [], []
        self.hazard_preds, self.hazard_targets = [], []
        self.severity_preds, self.severity_targets = [], []

    def update(self, total_loss, cls_loss, reg_loss, h_pred, h_true, s_pred, s_true):
        self.total_losses.append(total_loss)
        self.cls_losses.append(cls_loss)
        self.reg_losses.append(reg_loss)
        self.hazard_preds.extend(h_pred)
        self.hazard_targets.extend(h_true)
        self.severity_preds.extend(s_pred)
        self.severity_targets.extend(s_true)

    def get_summary(self):
        h_acc = accuracy_score(self.hazard_targets, self.hazard_preds)
        h_f1 = f1_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0)
        s_mse = mean_squared_error(self.severity_targets, self.severity_preds)
        return {
            'loss_total': np.mean(self.total_losses),
            'loss_cls': np.mean(self.cls_losses),
            'loss_reg': np.mean(self.reg_losses),
            'hazard_accuracy': h_acc, 'hazard_f1': h_f1,
            'severity_mse': s_mse, 'severity_rmse': np.sqrt(s_mse),
            'severity_mae': mean_absolute_error(self.severity_targets, self.severity_preds),
            'severity_r2': r2_score(self.severity_targets, self.severity_preds),
        }

    def get_per_class_metrics(self):
        prec = precision_score(self.hazard_targets, self.hazard_preds,
                               average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        rec = recall_score(self.hazard_targets, self.hazard_preds,
                           average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        f1 = f1_score(self.hazard_targets, self.hazard_preds,
                      average=None, labels=range(len(HAZARD_TYPES)), zero_division=0)
        support = np.bincount(self.hazard_targets, minlength=len(HAZARD_TYPES))
        rows = []
        for i, name in enumerate(HAZARD_TYPES):
            rows.append({'Hazard': name, 'Precision': prec[i], 'Recall': rec[i],
                         'F1-Score': f1[i], 'Support': support[i]})
        rows.append({'Hazard': 'Macro Avg',
                     'Precision': precision_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'Recall': recall_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'F1-Score': f1_score(self.hazard_targets, self.hazard_preds, average='macro', zero_division=0),
                     'Support': sum(support)})
        rows.append({'Hazard': 'Weighted Avg',
                     'Precision': precision_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'Recall': recall_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'F1-Score': f1_score(self.hazard_targets, self.hazard_preds, average='weighted', zero_division=0),
                     'Support': sum(support)})
        return pd.DataFrame(rows)

    def get_severity_error_by_quartile(self):
        targets = np.array(self.severity_targets)
        preds = np.array(self.severity_preds)
        if len(targets) == 0: return pd.DataFrame()
        quartiles = np.percentile(targets, [25, 50, 75])
        bins = [0, quartiles[0], quartiles[1], quartiles[2], 1.0]
        labels = ['Q1 (Low)', 'Q2 (Moderate)', 'Q3 (High)', 'Q4 (Severe)']
        bin_idx = np.digitize(targets, bins[1:-1])
        rows = []
        for q in range(4):
            mask = bin_idx == q
            if mask.sum() == 0: continue
            t_q, p_q = targets[mask], preds[mask]
            rows.append({'Severity Quartile': labels[q], 'N': int(mask.sum()),
                         'MAE': mean_absolute_error(t_q, p_q),
                         'RMSE': np.sqrt(mean_squared_error(t_q, p_q)),
                         'Mean Predicted': p_q.mean(), 'Mean Actual': t_q.mean()})
        return pd.DataFrame(rows)

    def get_confusion_matrix_normalized(self):
        cm = confusion_matrix(self.hazard_targets, self.hazard_preds, labels=range(len(HAZARD_TYPES)))
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        return np.nan_to_num(cm_norm)


# ============================================================================
# PUBLICATION FIGURES GENERATOR
# ============================================================================
class PublicationFigureGenerator:
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(os.path.join(output_dir, 'figures'), exist_ok=True)
        plt.rcParams.update({'font.size': 10, 'axes.labelsize': 12, 'axes.titlesize': 13,
                             'xtick.labelsize': 9, 'ytick.labelsize': 9,
                             'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})

    def plot_confusion_matrix(self, cm_normalized, title, filename):
        fig, ax = plt.subplots(figsize=(8, 7))
        sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=HAZARD_TYPES, yticklabels=HAZARD_TYPES, ax=ax)
        ax.set_xlabel('Predicted Hazard'); ax.set_ylabel('True Hazard'); ax.set_title(title)
        plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"  Saved: {path}")

    def plot_severity_scatter(self, targets, preds, r2, title, filename):
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(targets, preds, alpha=0.3, s=10, c='steelblue')
        lims = [0, 1]
        ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('Ground Truth Severity'); ax.set_ylabel('Predicted Severity')
        ax.set_title(f"{title}\nR²={r2:.4f}"); ax.legend(); ax.set_aspect('equal')
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"   Saved: {path}")

    def plot_spatial_heatmap(self, results_df, title, filename):
        if 'division' not in results_df.columns or 'season' not in results_df.columns: return
        pivot = results_df.pivot_table(values='accuracy', index='division', columns='season', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0.5, vmax=1.0, ax=ax)
        ax.set_title(title)
        plt.tight_layout()
        path = os.path.join(self.output_dir, 'figures', filename)
        plt.savefig(path); plt.close()
        print(f"   Saved: {path}")


# ============================================================================
# TRAINING & EVALUATION FUNCTIONS
# ============================================================================
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    metrics = EnhancedMetricsTracker()
    pbar = tqdm(loader, desc="Train", unit="batch")
    for tensors, cls_idx, severity, confidence, _ in pbar:
        tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
        optimizer.zero_grad()
        h_pred, s_pred = model(tensors)
        total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=TrainConfig.GRAD_CLIP)
        optimizer.step()
        metrics.update(total.item(), cls_l, reg_l,
                       h_pred.detach().argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                       s_pred.detach().cpu().numpy(), severity.cpu().numpy())
        pbar.set_postfix({'loss': f"{total.item():.4f}"})
    return metrics


def evaluate(model, loader, criterion, device, split_name="Val"):
    model.eval()
    metrics = EnhancedMetricsTracker()
    with torch.no_grad():
        pbar = tqdm(loader, desc=split_name, unit="batch")
        for tensors, cls_idx, severity, confidence, _ in pbar:
            tensors, cls_idx, severity, confidence = [t.to(device) for t in [tensors, cls_idx, severity, confidence]]
            h_pred, s_pred = model(tensors)
            total, cls_l, reg_l = criterion(h_pred, s_pred, cls_idx, severity, confidence)
            metrics.update(total.item(), cls_l, reg_l,
                           h_pred.argmax(1).cpu().numpy(), cls_idx.cpu().numpy(),
                           s_pred.cpu().numpy(), severity.cpu().numpy())
            pbar.set_postfix({'loss': f"{total.item():.4f}"})
    return metrics


def train_single_fold(fold_name, train_csv, val_csv, test_csv, num_classes, output_dir):
    print(f"\n{'─'*60}")
    print(f" {fold_name}")
    print(f"{'─'*60}")

    train_loader = DataLoader(MasterHDF5Dataset(train_csv, TrainConfig.MASTER_H5_PATH, True),
                              batch_size=TrainConfig.BATCH_SIZE, shuffle=True,
                              num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(MasterHDF5Dataset(val_csv, TrainConfig.MASTER_H5_PATH, False),
                            batch_size=TrainConfig.BATCH_SIZE, shuffle=False,
                            num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(MasterHDF5Dataset(test_csv, TrainConfig.MASTER_H5_PATH, False),
                             batch_size=TrainConfig.BATCH_SIZE, shuffle=False,
                             num_workers=TrainConfig.NUM_WORKERS, pin_memory=True)

    print(f"  Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")

    model = HazardNetCNN(15, num_classes).to(TrainConfig.DEVICE)
    criterion = HomoscedasticMTLLoss().to(TrainConfig.DEVICE)
    optimizer = AdamW([{'params': model.parameters()}, {'params': criterion.log_vars}],
                      lr=TrainConfig.LEARNING_RATE, weight_decay=TrainConfig.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=TrainConfig.NUM_EPOCHS, eta_min=1e-6)

    best_val_loss, patience_counter, best_epoch = float('inf'), 0, 0
    safe_name = fold_name.replace('/', '_').replace(' ', '_')
    ckpt_path = os.path.join(output_dir, f'{safe_name}_best.pt')
    fig_gen = PublicationFigureGenerator(output_dir)

    # ── resume, if a previous attempt died inside this fold ───────────────────
    # `output_dir` is on Drive, so a recycled session left its state behind. A relaunch
    # (model_retrain_watch.yml sets `resume: true` in run_config.json) picks the fold up
    # at the epoch it reached, with the optimizer, the cosine schedule and the RNGs where
    # they were. Restoring weights alone is not enough: it silently restarts the learning
    # rate schedule, and the run then converges to something nobody validated.
    state_path = os.path.join(output_dir, f'{safe_name}_resume.pt')
    start_epoch = 0
    if TrainConfig.RESUME and os.path.exists(state_path):
        try:
            state = torch.load(state_path, map_location=TrainConfig.DEVICE)
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            start_epoch = int(state['epoch']) + 1
            best_val_loss, best_epoch = state['best_val_loss'], state['best_epoch']
            patience_counter = int(state.get('patience_counter', 0))
            torch.set_rng_state(state['rng'].cpu())
            np.random.set_state(state['np_rng'])
            RESUMED_FROM.append(f'{safe_name}@epoch{start_epoch}')
            print(f"  ↻ resumed {fold_name} at epoch {start_epoch} "
                  f"(best {best_epoch}, val_loss {best_val_loss:.4f})")
        except Exception as exc:
            # A truncated state file on a FUSE mount is a real case, not a hypothetical:
            # the session can die mid-write. Starting over is correct; guessing is not.
            print(f"  ⚠ could not resume {fold_name} ({exc}); starting from epoch 0")
            start_epoch = 0
    elif os.path.exists(state_path):
        print(f"  · ignoring the resume state at {state_path} (resume=False)")

    # W&B Init per fold

    run = None
    if WANDB_ENABLED:
        try:
            run = wandb.init(project='hazardnet', name=safe_name, reinit=True,
                             config={'strategy': fold_name, 'batch_size': TrainConfig.BATCH_SIZE})
        except: pass

    for epoch in range(start_epoch, TrainConfig.NUM_EPOCHS):
        train_m = train_epoch(model, train_loader, optimizer, criterion, TrainConfig.DEVICE)
        val_m = evaluate(model, val_loader, criterion, TrainConfig.DEVICE, "Val")
        scheduler.step()
        ts, vs = train_m.get_summary(), val_m.get_summary()

        if vs['loss_total'] < best_val_loss:
            best_val_loss = vs['loss_total']; patience_counter = 0; best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= TrainConfig.PATIENCE:
                print(f"  ⏹️ Early stopping at epoch {epoch+1} (best: {best_epoch})"); break

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{TrainConfig.NUM_EPOCHS} | "
                  f"Train: {ts['loss_total']:.4f} Acc:{ts['hazard_accuracy']:.3f} | "
                  f"Val: {vs['loss_total']:.4f} Acc:{vs['hazard_accuracy']:.3f} RMSE:{vs['severity_rmse']:.4f}")

        if run:
            wandb.log({'train_loss': ts['loss_total'], 'val_loss': vs['loss_total'],
                       'train_acc': ts['hazard_accuracy'], 'val_acc': vs['hazard_accuracy']})

        # ── survive a recycled session ────────────────────────────────────────
        # Two writes per epoch, both to Drive. The resumable state is what lets a
        # relaunched attempt continue this fold instead of restarting it; the heartbeat is
        # what tells the watcher the session is alive. Together they are the difference
        # between a free-tier recycle costing one epoch and costing eight hours.
        if TrainConfig.CHECKPOINT_EVERY and (epoch + 1) % TrainConfig.CHECKPOINT_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'best_epoch': best_epoch,
                'patience_counter': patience_counter,
                'rng': torch.get_rng_state(),
                'np_rng': np.random.get_state(),
            }, state_path)
        beat('training', fold=fold_name, epoch=epoch + 1, epochs=TrainConfig.NUM_EPOCHS,
             val_loss=float(vs['loss_total']), val_acc=float(vs['hazard_accuracy']))


    # TEST EVALUATION WITH FULL METRICS
    model.load_state_dict(torch.load(ckpt_path))
    test_metrics = evaluate(model, test_loader, criterion, TrainConfig.DEVICE, "Test")
    test_summary = test_metrics.get_summary()

    print(f"   Test: Acc={test_summary['hazard_accuracy']:.4f} F1={test_summary['hazard_f1']:.4f} "
          f"RMSE={test_summary['severity_rmse']:.4f} R²={test_summary['severity_r2']:.4f}")

    # Generate Publication Tables & Figures
    per_class = test_metrics.get_per_class_metrics()
    per_class.to_csv(os.path.join(output_dir, f'{safe_name}_per_class.csv'), index=False)

    sev_quartile = test_metrics.get_severity_error_by_quartile()
    if not sev_quartile.empty:
        sev_quartile.to_csv(os.path.join(output_dir, f'{safe_name}_severity_quartile.csv'), index=False)

    fig_gen.plot_confusion_matrix(test_metrics.get_confusion_matrix_normalized(),
                                  f'Confusion Matrix: {fold_name}', f'{safe_name}_confusion_matrix.png')
    fig_gen.plot_severity_scatter(test_metrics.severity_targets, test_metrics.severity_preds,
                                  test_summary['severity_r2'], f'Severity: {fold_name}', f'{safe_name}_severity_scatter.png')

    if run: wandb.finish()

    # The fold is finished, so its resume state is not. Leaving it behind would let a later
    # attempt "resume" a fold that already completed, and report the same fold twice.
    if os.path.exists(state_path):
        try:
            os.remove(state_path)
        except OSError:
            pass

    return {
        'fold': fold_name, 'accuracy': test_summary['hazard_accuracy'],
        'f1': test_summary['hazard_f1'], 'rmse': test_summary['severity_rmse'],
        'mae': test_summary['severity_mae'], 'r2': test_summary['severity_r2'],
        'n_test': len(test_loader.dataset),
    }


# ============================================================================
# STRATEGY RUNNERS
# ============================================================================
def run_event_kfold(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'event_kfold')
    results = []
    for fold_idx in range(5):
        fd = os.path.join(base, f'fold_{fold_idx}')
        if not os.path.exists(fd): print(f"   Skipping fold_{fold_idx}"); continue
        r = train_single_fold(f'event_kfold_fold{fold_idx}',
                              os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        results.append(r)
    return results

def run_spatial_lodo(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'spatial_lodo')
    results = []
    fold_dirs = sorted(glob.glob(os.path.join(base, 'lodo_division_*')))
    print(f"  Found {len(fold_dirs)} LODO division folds")
    for fd in fold_dirs:
        fn = os.path.basename(fd)
        r = train_single_fold(fn, os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        r['division'] = fn.replace('lodo_division_', '')
        results.append(r)
    return results

def run_temporal(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'temporal_split')
    if not os.path.exists(os.path.join(base, 'train_events.csv')):
        print("   Temporal split not found"); return []
    r = train_single_fold('temporal_season_adaptive',
                          os.path.join(base, 'train_events.csv'), os.path.join(base, 'val_events.csv'),
                          os.path.join(base, 'test_events.csv'), num_classes, output_dir)
    return [r]

def run_spatio_temporal(num_classes, output_dir):
    base = os.path.join(TrainConfig.EXPERIMENTAL_DIR, 'spatio_temporal')
    results = []
    fold_dirs = sorted(glob.glob(os.path.join(base, 'st_*')))
    print(f"  Found {len(fold_dirs)} spatio-temporal folds")
    for fd in fold_dirs:
        fn = os.path.basename(fd)
        r = train_single_fold(fn, os.path.join(fd, 'train_events.csv'), os.path.join(fd, 'val_events.csv'),
                              os.path.join(fd, 'test_events.csv'), num_classes, output_dir)
        parts = fn.replace('st_', '').rsplit('_', 1)
        if len(parts) == 2: r['division'], r['season'] = parts[0], parts[1]
        results.append(r)
    return results


# ============================================================================
# MAIN ORCHESTRATOR
# ============================================================================
STRATEGY_MAP = {
    'event_kfold': ('Event-Based 5-Fold CV', run_event_kfold),
    'spatial_lodo': ('Spatial LODO (Division-Level)', run_spatial_lodo),
    'temporal': ('Temporal Split (Season-Adaptive)', run_temporal),
    'spatio_temporal': ('Spatio-Temporal (Division×Season×Era)', run_spatio_temporal),
}

# ═══════════════════════════════════════════════════════════
# SET YOUR STRATEGY HERE (for Kaggle notebook execution)
# ═══════════════════════════════════════════════════════════
# Options: event_kfold, spatial_lodo, temporal, spatio_temporal, all.
# The choice comes from the run contract cell — run_config.json when Actions launched this
# notebook, 'event_kfold' when a human ran it by hand. Editing it here would be overwritten.
STRATEGY = RUN_STRATEGY
# ═══════════════════════════════════════════════════════════

def main():
    print("=" * 80)
    print("  HAZARDNET UNIFIED EXPERIMENTAL TRAINING (Q1 Journal Edition)")
    print("=" * 80)

    with open(TrainConfig.CONFIG_PATH, 'r') as f:
        config = json.load(f)
    num_classes = config['n_classes']

    print(f"  Classes ({num_classes}): {config['hazard_types']}")
    print(f"  Master HDF5: {TrainConfig.MASTER_H5_PATH}")
    print(f"  Device: {TrainConfig.DEVICE}")
    print(f"  W&B Enabled: {WANDB_ENABLED}")

    strategies = list(STRATEGY_MAP.items()) if STRATEGY == 'all' else [(STRATEGY, STRATEGY_MAP[STRATEGY])]
    all_strategy_results = {}

    for strat_key, (strat_name, strat_fn) in strategies:
        print(f"\n{'='*80}")
        print(f"STRATEGY: {strat_name.upper()}")
        print(f"{'='*80}")

        strat_output = os.path.join(TrainConfig.OUTPUT_DIR, strat_key)
        os.makedirs(strat_output, exist_ok=True)

        results = strat_fn(num_classes, strat_output)
        all_strategy_results[strat_key] = results

        if results:
            accs = [r['accuracy'] for r in results]
            f1s = [r['f1'] for r in results]
            rmses = [r['rmse'] for r in results]
            r2s = [r['r2'] for r in results]
            print(f"\n  {strat_name} Summary ({len(results)} folds):")
            print(f"     Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
            print(f"     F1-Score: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
            print(f"     RMSE:     {np.mean(rmses):.4f} ± {np.std(rmses):.4f}")
            print(f"     R²:       {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")

            df = pd.DataFrame([{k: v for k, v in r.items()} for r in results])
            df.to_csv(os.path.join(strat_output, f'{strat_key}_results.csv'), index=False)

            if strat_key in ['spatial_lodo', 'spatio_temporal']:
                fig_gen = PublicationFigureGenerator(strat_output)
                fig_gen.plot_spatial_heatmap(df, f'{strat_name} Accuracy', f'{strat_key}_spatial_heatmap.png')

    # CROSS-STRATEGY COMPARISON
    print(f"\n{'='*80}")
    print("CROSS-STRATEGY COMPARISON (IEEE TGRS Table II)")
    print(f"{'='*80}")

    comparison_rows = []
    for strat_key, (strat_name, _) in STRATEGY_MAP.items():
        results = all_strategy_results.get(strat_key, [])
        if results:
            accs = [r['accuracy'] for r in results]
            f1s = [r['f1'] for r in results]
            rmses = [r['rmse'] for r in results]
            r2s = [r['r2'] for r in results]
            comparison_rows.append({
                'Strategy': strat_name, 'N_Folds': len(results),
                'Accuracy': f"{np.mean(accs):.4f} ± {np.std(accs):.4f}",
                'F1-Score': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
                'RMSE': f"{np.mean(rmses):.4f} ± {np.std(rmses):.4f}",
                'R²': f"{np.mean(r2s):.4f} ± {np.std(r2s):.4f}",
                'Total_Test_Events': sum(r['n_test'] for r in results),
            })

    if comparison_rows:
        df_comp = pd.DataFrame(comparison_rows)
        print(df_comp.to_string(index=False))
        comp_path = os.path.join(TrainConfig.OUTPUT_DIR, 'cross_strategy_comparison.csv')
        df_comp.to_csv(comp_path, index=False)
        print(f"\nComparison saved to: {comp_path}")

    print(f"\nAll experimental training complete!")
    print(f"   Results: {TrainConfig.OUTPUT_DIR}")
    # Returned so the publish cell can put the numbers the training actually measured into
    # the manifest. A metric transcribed by hand eventually stops matching the run that
    # produced it.
    return all_strategy_results


TRAINING_RESULTS = main()
beat('trained', message=f'{sum(len(rows) for rows in (TRAINING_RESULTS or {}).values())} folds complete')

In [ ]:
# Choose the checkpoint to convert from the results the training actually produced.
#
# This cell used to assert one hardcoded path (`event_kfold/event_kfold_fold2_best.pt`),
# so training any other strategy — or an event fold that early-stopped differently —
# failed the notebook at the last mile, after eight hours of GPU, or worse: converted a
# checkpoint left behind by a previous run.
FOLD_RESULTS = [dict(row, strategy=strategy)
                for strategy, rows in (TRAINING_RESULTS or {}).items() for row in rows]
assert FOLD_RESULTS, 'training reported no folds, so there is nothing to convert'

BEST_FOLD = max(FOLD_RESULTS, key=lambda row: (row.get('accuracy') or 0.0))
_safe = str(BEST_FOLD['fold']).replace('/', '_').replace(' ', '_')
BEST_PT = os.path.join(TrainConfig.OUTPUT_DIR, BEST_FOLD['strategy'], f'{_safe}_best.pt')

assert os.path.exists(BEST_PT), (
    f'training reported fold {BEST_FOLD["fold"]} but did not leave {BEST_PT}; '
    f'{TrainConfig.OUTPUT_DIR} contains: {sorted(os.listdir(TrainConfig.OUTPUT_DIR))}'
)
print(f'converting {BEST_PT} ({os.path.getsize(BEST_PT) / 1e6:.2f} MB)')
print(f'  fold {BEST_FOLD["fold"]} | strategy {BEST_FOLD["strategy"]} | '
      f'accuracy {BEST_FOLD.get("accuracy"):.4f} | f1 {BEST_FOLD.get("f1"):.4f}')
print(f'  {len(FOLD_RESULTS)} folds trained across {len(TRAINING_RESULTS)} strategy/strategies')

In [ ]:
import os, json, shutil
os.makedirs(BUNDLE_DIR, exist_ok=True)

In [ ]:
import os
import json
import glob
import subprocess
import shutil
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import h5py
from tqdm import tqdm


class DeployConfig:
    # From the cells above rather than retyped. This used to hardcode one fold's
    # checkpoint, so training any other strategy quietly converted the *previous* run's
    # weights — the least visible way there is to ship a stale model.
    BEST_CHECKPOINT = BEST_PT
    # The Drive copy, deliberately: the representative dataset and the parity test both read
    # the fold CSVs that live beside the tensor, and those are not copied to local disk.
    MASTER_H5_PATH = str(TENSOR_SRC)
    CONFIG_PATH = os.path.join(str(TENSOR_DIR), 'dataset_config.json')
    # The normalization statistics travel with the tensor; with no Drive they come from the
    # Kaggle download, and a run that cannot find them must not fall back to invented
    # defaults — a bundle normalized with the wrong statistics is worse than no bundle.
    NORMALIZATION_STATS_PATH = next(
        (str(path) for path in (TENSOR_DIR / 'normalization_stats.json',
                                TENSOR_DIR.parent / 'normalization_stats.json',
                                DRIVE_ROOT / 'tensors_output' / 'normalization_stats.json')
         if path.is_file()),
        str(TENSOR_DIR / 'normalization_stats.json'))
    OUTPUT_DIR = str(Path(BUNDLE_DIR).parent)

    IN_CHANNELS = 15
    NUM_HAZARDS = 8
    INPUT_SHAPE = (1, 15, 10, 64, 64)
    NUM_REPRESENTATIVE_SAMPLES = 100

    BAND_NAMES = [
        'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR',
        'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp',
        'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
    ]

os.makedirs(DeployConfig.OUTPUT_DIR, exist_ok=True)


class NormalizationStats:
    """Robust Normalization Stats Loader that adapts to various JSON schemas."""
    def __init__(self, stats_path: str, band_names: list):
        print(f"Loading normalization stats from: {stats_path}")
        if not os.path.exists(stats_path):
            raise FileNotFoundError(f"Normalization stats file not found: {stats_path}")

        with open(stats_path, 'r') as f:
            self.stats = json.load(f)

        self.band_names = band_names
        self.means_list = []
        self.stds_list = []

        self._parse_and_validate()

        self.means = np.array(self.means_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.array(self.stds_list, dtype=np.float32).reshape(1, -1, 1, 1, 1)
        self.stds = np.where(self.stds < 1e-8, 1.0, self.stds)

        print(f"  [OK] Successfully loaded normalization stats for {len(self.band_names)} bands")

    def _parse_and_validate(self):
        data = self.stats

        if isinstance(data, dict):
            for nested_key in ['per_band', 'bands', 'stats', 'band_stats']:
                if nested_key in data and isinstance(data[nested_key], (dict, list)):
                    data = data[nested_key]
                    break

        if isinstance(data, dict) and any(k in data for k in ['mean', 'means']) and any(k in data for k in ['std', 'stds']):
            m_key = 'mean' if 'mean' in data else 'means'
            s_key = 'std' if 'std' in data else 'stds'
            means_data, stds_data = data[m_key], data[s_key]

            if isinstance(means_data, list) and isinstance(stds_data, list):
                if len(means_data) == len(self.band_names):
                    self.means_list = [float(m) for m in means_data]
                    self.stds_list = [float(s) for s in stds_data]
                    return
                else:
                    raise ValueError(f"Stats list length ({len(means_data)}) != expected bands ({len(self.band_names)})")

            elif isinstance(means_data, dict) and isinstance(stds_data, dict):
                means_lower = {str(k).lower(): v for k, v in means_data.items()}
                stds_lower = {str(k).lower(): v for k, v in stds_data.items()}
                for i, b in enumerate(self.band_names):
                    b_lower, b_idx = b.lower(), str(i)
                    if b_lower in means_lower and b_lower in stds_lower:
                        self.means_list.append(float(means_lower[b_lower]))
                        self.stds_list.append(float(stds_lower[b_lower]))
                    elif b_idx in means_lower and b_idx in stds_lower:
                        self.means_list.append(float(means_lower[b_idx]))
                        self.stds_list.append(float(stds_lower[b_idx]))
                    else:
                        raise ValueError(f"Could not find mean/std for band '{b}' in stats dict")
                return

        if isinstance(data, dict):
            key_map = {str(k).lower(): v for k, v in data.items() if isinstance(v, dict)}
            missing = []
            for i, band in enumerate(self.band_names):
                band_lower, band_idx = band.lower(), str(i)
                target = key_map.get(band_lower) or key_map.get(band_idx)
                if target is not None:
                    m_val = target.get('mean', target.get('means'))
                    s_val = target.get('std', target.get('stds'))
                    if m_val is not None and s_val is not None:
                        self.means_list.append(float(m_val))
                        self.stds_list.append(float(s_val))
                        continue
                missing.append(band)
            if not missing:
                return
            raise ValueError(f"Normalization stats missing bands: {missing}.")

        if isinstance(data, list) and all(isinstance(x, dict) for x in data):
            band_map = {}
            for entry in data:
                b_name = entry.get('band', entry.get('name', entry.get('band_name')))
                if b_name is not None:
                    band_map[str(b_name).lower()] = entry
            for i, band in enumerate(self.band_names):
                entry = band_map.get(band.lower()) or band_map.get(str(i))
                if entry and 'mean' in entry and 'std' in entry:
                    self.means_list.append(float(entry['mean']))
                    self.stds_list.append(float(entry['std']))
                else:
                    raise ValueError(f"Missing stats entry for band '{band}' in list of stats.")
            return

        raise ValueError("Unrecognized normalization stats JSON structure.")

    def normalize(self, tensor: np.ndarray) -> np.ndarray:
        if tensor.ndim == 4:
            means, stds = self.means[0], self.stds[0]
        elif tensor.ndim == 5:
            means, stds = self.means, self.stds
        else:
            raise ValueError(f"Expected 4D or 5D tensor, got {tensor.ndim}D")
        return (tensor.astype(np.float32) - means) / stds

    def to_dict(self) -> dict:
        return {
            'means': {b: float(self.means_list[i]) for i, b in enumerate(self.band_names)},
            'stds': {b: float(self.stds_list[i]) for i, b in enumerate(self.band_names)},
            'band_order': self.band_names,
            'normalization_type': 'z_score',
        }


class DepthwiseSeparableConv3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_channels, in_channels, kernel_size, padding=padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm3d(out_channels)

    def forward(self, x):
        return self.bn(self.pointwise(self.depthwise(x)))


class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x).unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        return x * w


class HazardNetCNN(nn.Module):
    def __init__(self, in_channels=15, num_hazards=8):
        super().__init__()
        self.block1 = nn.Sequential(DepthwiseSeparableConv3d(in_channels, 32), nn.ReLU(True), SEBlock3D(32), nn.MaxPool3d((1, 2, 2)))
        self.block2 = nn.Sequential(DepthwiseSeparableConv3d(32, 64), nn.ReLU(True), SEBlock3D(64), nn.MaxPool3d((2, 2, 2)))
        self.block3 = nn.Sequential(DepthwiseSeparableConv3d(64, 128), nn.ReLU(True), SEBlock3D(128), nn.MaxPool3d((1, 2, 2)))
        self.block4 = nn.Sequential(DepthwiseSeparableConv3d(128, 256), nn.ReLU(True), SEBlock3D(256), nn.MaxPool3d((1, 2, 2)))
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.shared_fc = nn.Sequential(nn.Linear(256, 128), nn.ReLU(True), nn.Dropout(0.3))
        self.hazard_head = nn.Linear(128, num_hazards)
        self.severity_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(True), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.global_pool(x).view(x.size(0), -1)
        x = self.shared_fc(x)
        return self.hazard_head(x), self.severity_head(x).squeeze(1)


def load_model_and_stats():
    print("=" * 70)
    print("STEP 1: Loading Model & Pre-computed Normalization Stats")
    print("=" * 70)
    norm_stats = NormalizationStats(DeployConfig.NORMALIZATION_STATS_PATH, DeployConfig.BAND_NAMES)

    model = HazardNetCNN(DeployConfig.IN_CHANNELS, DeployConfig.NUM_HAZARDS)
    state_dict = torch.load(DeployConfig.BEST_CHECKPOINT, map_location='cpu')
    model.load_state_dict(state_dict)
    model.eval()

    params = sum(p.numel() for p in model.parameters())
    print(f"  [OK] Model loaded: {params:,} params (~{params * 4 / 1024**2:.2f} MB FP32)")
    return model, norm_stats


def export_to_onnx(model):
    print("\n" + "=" * 70)
    print("STEP 2: Exporting to ONNX")
    print("=" * 70)
    output_path = os.path.join(DeployConfig.OUTPUT_DIR, 'hazardnet.onnx')
    dummy_input = torch.randn(*DeployConfig.INPUT_SHAPE)

    print("  Tracing model with torch.jit.trace to bypass onnxscript registry bugs...")
    try:
        export_target = torch.jit.trace(model, dummy_input)
    except Exception as e:
        print(f"  [WARN] Tracing warning ({e}), falling back to PyTorch model")
        export_target = model

    torch.onnx.export(
        export_target, dummy_input, output_path,
        export_params=True, opset_version=17, do_constant_folding=True,
        input_names=['input'],
        output_names=['hazard_logits', 'severity_pred'],
        dynamic_axes={'input': {0: 'batch_size'}, 'hazard_logits': {0: 'batch_size'}, 'severity_pred': {0: 'batch_size'}},
        dynamo=False
    )

    import onnx
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print(f"  [OK] ONNX exported: {output_path} ({os.path.getsize(output_path) / 1024**2:.2f} MB)")
    return output_path


def create_representative_dataset(norm_stats: NormalizationStats):
    print("\n" + "=" * 70)
    print(f"STEP 3: Creating Representative Dataset ({DeployConfig.NUM_REPRESENTATIVE_SAMPLES} samples)")
    print("=" * 70)

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    train_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/train_events.csv')))
    if not train_csvs:
        raise FileNotFoundError(f"No train CSVs found in {event_kfold_dir}")

    samples_per_fold = max(1, DeployConfig.NUM_REPRESENTATIVE_SAMPLES // len(train_csvs))
    all_event_ids = []
    for csv_path in train_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:DeployConfig.NUM_REPRESENTATIVE_SAMPLES]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    samples = []
    for eid in tqdm(all_event_ids, desc="Loading & normalizing"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        samples.append(normalized[np.newaxis, ...])
    h5f.close()

    print(f"  [OK] Created {len(samples)} normalized representative samples")
    return samples


def convert_to_tflite(onnx_path, representative_samples):
    print("\n" + "=" * 70)
    print("STEP 4: Converting to TFLite")
    print("=" * 70)

    # Install onnx2tf if not already installed
    try:
        import onnx2tf
    except ImportError:
        print("  Installing onnx2tf...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx2tf'], check=True, capture_output=True, text=True)
        print("  onnx2tf installed successfully.")

    import tensorflow as tf

    tflite_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'tflite')
    os.makedirs(tflite_dir, exist_ok=True)
    tf_saved_model_dir = os.path.join(tflite_dir, 'tf_saved_model')

    print("  Converting ONNX -> TF SavedModel / TFLite...")

    # Robust CLI execution using sys.executable to avoid PATH issues in Kaggle/Colab
    cmd = [sys.executable, '-m', 'onnx2tf', '-i', onnx_path, '-o', tf_saved_model_dir, '-osd', '-nuo']
    print(f"  Running command: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Print stdout and stderr for debugging
    print("onnx2tf stdout:")
    print(result.stdout)
    print("onnx2tf stderr:")
    print(result.stderr)

    # Raise an exception if onnx2tf failed
    result.check_returncode()
    print("  [OK] onnx2tf command executed successfully.")

    def _find_saved_model(path):
        for root, _, files in os.walk(path):
            if 'saved_model.pb' in files: return root
        return None

    saved_pb_path = _find_saved_model(tf_saved_model_dir)
    fp32_model_bytes = None

    if saved_pb_path:
        print(f"  [OK] Found TF SavedModel at: {saved_pb_path}")
        print("  Converting SavedModel to Pure FP32 TFLite...")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_pb_path)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32
        fp32_model_bytes = converter.convert()
    else:
        # Fallback: onnx2tf often outputs .tflite directly if SavedModel generation fails
        direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*float32.tflite"))
        if not direct_tflite_files:
            direct_tflite_files = glob.glob(os.path.join(tf_saved_model_dir, "*.tflite"))

        if direct_tflite_files:
            saved_pb_path = direct_tflite_files[0]
            print(f"  [OK] Found direct FP32 TFLite model: {saved_pb_path}")
            with open(saved_pb_path, 'rb') as f:
                fp32_model_bytes = f.read()
        else:
            raise RuntimeError(f"TF SavedModel or TFLite not created at '{tf_saved_model_dir}'.")

    fp32_path = os.path.join(tflite_dir, 'hazardnet_fp32.tflite')
    with open(fp32_path, 'wb') as f:
        f.write(fp32_model_bytes)

    print("\n  [INFO] ARCHITECTURE LIMITATION DETECTED")
    print("  TensorFlow Lite's native 'CONV_3D' kernel strictly requires FLOAT32 tensors.")
    print("  Applying Optimize.DEFAULT (INT8) causes a runtime crash in conv3d.cc.")
    print("  To guarantee edge compatibility, INT8 quantization is safely bypassed.")

    # Save FP32 as the final optimized model for edge deployment
    int8_path = os.path.join(tflite_dir, 'hazardnet_optimized_fp32.tflite')
    with open(int8_path, 'wb') as f:
        f.write(fp32_model_bytes)

    fp32_mb = len(fp32_model_bytes) / (1024 ** 2)
    print(f"\n  TFLite Results:")
    print(f"     Model Size: {fp32_mb:.2f} MB (Pure FP32)")
    print(f"     {'[OK] UNDER 150 MB TARGET' if fp32_mb < 150 else '[WARN] EXCEEDS 150 MB'}")

    return fp32_path, int8_path


def golden_parity_test(model, norm_stats, int8_path, n_samples=50):
    print("\n" + "=" * 70)
    print(f"STEP 5: Golden Parity Test ({n_samples} samples)")
    print("=" * 70)

    import tensorflow as tf

    event_kfold_dir = os.path.dirname(DeployConfig.MASTER_H5_PATH)
    test_csvs = sorted(glob.glob(os.path.join(event_kfold_dir, 'event_kfold/fold_*/test_events.csv')))
    if not test_csvs:
        print("  [WARN] No test CSVs found, skipping parity test")
        return 0.0, 0.0

    samples_per_fold = max(1, n_samples // len(test_csvs))
    all_event_ids = []
    for csv_path in test_csvs:
        fold_df = pd.read_csv(csv_path)
        fold_sample = fold_df.sample(n=min(samples_per_fold, len(fold_df)), random_state=42)
        all_event_ids.extend(fold_sample['event_id'].astype(str).tolist())
    all_event_ids = all_event_ids[:n_samples]

    h5f = h5py.File(DeployConfig.MASTER_H5_PATH, 'r')
    interpreter = tf.lite.Interpreter(model_path=int8_path)
    interpreter.allocate_tensors()
    inp_details = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()

    tflite_input_shape = inp_details['shape']
    # Detect if TFLite expects NDHWC (1, 10, 64, 64, 15) instead of NCDHW (1, 15, 10, 64, 64)
    needs_transpose = (len(tflite_input_shape) == 5 and tflite_input_shape[1] != DeployConfig.IN_CHANNELS and tflite_input_shape[4] == DeployConfig.IN_CHANNELS)

    print(f"  [INFO] TFLite expected input shape: {tuple(tflite_input_shape)}")
    print(f"  [INFO] Transpose required (NCDHW -> NDHWC): {needs_transpose}")

    model.eval()
    hazard_agreements = 0
    severity_diffs = []

    for eid in tqdm(all_event_ids, desc="Parity test"):
        raw_tensor = h5f[f'tensors/{eid}'][:]
        normalized = norm_stats.normalize(raw_tensor)
        batch = normalized[np.newaxis, ...]

        with torch.no_grad():
            pt_hazard, pt_severity = model(torch.from_numpy(batch))
        pt_class = pt_hazard.argmax(dim=1).item()

        tflite_batch = np.transpose(batch, (0, 2, 3, 4, 1)) if needs_transpose else batch
        tflite_input = tflite_batch.astype(inp_details['dtype'])
        interpreter.set_tensor(inp_details['index'], tflite_input)
        interpreter.invoke()

        # Robust output extraction by shape rather than strict index
        tf_hazard, tf_severity = None, None
        for out in out_details:
            out_shape = out['shape']
            if len(out_shape) == 2 and out_shape[1] == DeployConfig.NUM_HAZARDS:
                tf_hazard = interpreter.get_tensor(out['index'])[0]
            elif len(out_shape) <= 2 and (out_shape[-1] == 1 or out_shape == (1,)):
                tf_severity = interpreter.get_tensor(out['index'])[0]
                if isinstance(tf_severity, np.ndarray):
                    tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        # Fallback if shape matching failed
        if tf_hazard is None or tf_severity is None:
            tf_hazard = interpreter.get_tensor(out_details[0]['index'])[0]
            tf_severity = interpreter.get_tensor(out_details[1]['index'])[0]
            if isinstance(tf_severity, np.ndarray):
                tf_severity = tf_severity.item() if tf_severity.size == 1 else tf_severity[0]

        if pt_class == int(np.argmax(tf_hazard)):
            hazard_agreements += 1
        severity_diffs.append(abs(pt_severity.item() - float(tf_severity)))

    h5f.close()
    agreement_pct = hazard_agreements / len(all_event_ids) * 100
    mean_sev_diff = np.mean(severity_diffs)

    print(f"\n  Parity Results:")
    print(f"     Hazard agreement: {agreement_pct:.1f}%")
    print(f"     Severity MAE (PT vs TFLite): {mean_sev_diff:.4f}")
    # The gate belongs to the contract, not to this cell: `retrain_state` enforces the same
    # number when it validates the manifest, so a conversion intake would refuse fails here
    # instead — eight hours earlier, with a log to read. Warning and carrying on used to be
    # the behaviour, which published a bundle that had changed predictions and let the pull
    # request be the first place anybody noticed.
    if agreement_pct < rs.PARITY_GATE_PCT:
        raise SystemExit(
            f'parity gate failed: TFLite agrees with PyTorch on {agreement_pct:.2f}% of hazards '
            f'(gate {rs.PARITY_GATE_PCT}%). A conversion that changes predictions is not a '
            'conversion, so nothing is published.'
        )
    print(f"     Status: [OK] PASSED (gate {rs.PARITY_GATE_PCT}%)")
    return agreement_pct, mean_sev_diff


def create_deployment_bundle(norm_stats: NormalizationStats, int8_path, fp32_path):
    print("\n" + "=" * 70)
    print("STEP 6: Creating Deployment Bundle")
    print("=" * 70)

    bundle_dir = os.path.join(DeployConfig.OUTPUT_DIR, 'deployment_bundle')
    os.makedirs(bundle_dir, exist_ok=True)

    for src_path, dst_name in [(int8_path, 'hazardnet_int8.tflite'), (fp32_path, 'hazardnet_fp32.tflite')]:
        if src_path and os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(bundle_dir, dst_name))

    if os.path.exists(DeployConfig.CONFIG_PATH):
        with open(DeployConfig.CONFIG_PATH, 'r') as f:
            config = json.load(f)
        labels = {str(i): h for i, h in enumerate(config.get('hazard_types', []))}
    else:
        labels = {str(i): f"Hazard_{i}" for i in range(DeployConfig.NUM_HAZARDS)}

    with open(os.path.join(bundle_dir, 'labels.json'), 'w') as f:
        json.dump(labels, f, indent=2)

    preprocessing_config = {
        'normalization': norm_stats.to_dict(),
        'input_shape': list(DeployConfig.INPUT_SHAPE),
        'num_hazards': DeployConfig.NUM_HAZARDS,
        'outputs': {'hazard_logits': 'index_0', 'severity_pred': 'index_1'},
    }
    with open(os.path.join(bundle_dir, 'preprocessing_config.json'), 'w') as f:
        json.dump(preprocessing_config, f, indent=2)

    inference_script = '''#!/usr/bin/env python3
"""HazardNet Edge Inference with Pre-computed Normalization"""
import numpy as np, tensorflow as tf, json, time, sys

def load_normalization_stats(config_path='preprocessing_config.json'):
    with open(config_path) as f: config = json.load(f)
    norm = config['normalization']
    means = np.array([norm['means'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    stds = np.array([norm['stds'][b] for b in norm['band_order']], dtype=np.float32).reshape(1, -1, 1, 1, 1)
    return means, np.where(stds < 1e-8, 1.0, stds)

def normalize(raw_tensor, means, stds):
    return (raw_tensor.astype(np.float32) - means) / stds

def predict(tflite_path, raw_tensor, means, stds, labels_path='labels.json'):
    normalized = normalize(raw_tensor, means, stds)

    # Transpose NCDHW -> NDHWC for TFLite if necessary
    if normalized.shape[1] == 15 and normalized.shape[2] == 10:
        normalized = np.transpose(normalized, (0, 2, 3, 4, 1))

    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    outs = interp.get_output_details()

    start = time.perf_counter()
    interp.set_tensor(inp['index'], normalized.astype(inp['dtype']))
    interp.invoke()
    latency = (time.perf_counter() - start) * 1000

    # Robust output extraction by shape
    hazard, severity = None, None
    for out in outs:
        if len(out['shape']) == 2 and out['shape'][1] == 8:
            hazard = interp.get_tensor(out['index'])[0]
        elif len(out['shape']) <= 2:
            severity = interp.get_tensor(out['index'])[0]

    if hazard is None: hazard = interp.get_tensor(outs[0]['index'])[0]
    if severity is None: severity = interp.get_tensor(outs[1]['index'])[0]
    if isinstance(severity, np.ndarray):
        severity = severity.item() if severity.size == 1 else severity[0]

    with open(labels_path) as f: labels = json.load(f)
    cls = int(np.argmax(hazard))
    return {
        'hazard': labels[str(cls)],
        'confidence': float(np.max(hazard)),
        'severity': float(severity),
        'latency_ms': latency
    }

if __name__ == '__main__':
    model = sys.argv[1] if len(sys.argv) > 1 else 'hazardnet_int8.tflite'
    means, stds = load_normalization_stats()
    r = predict(model, np.random.randn(1, 15, 10, 64, 64).astype(np.float32), means, stds)
    print(f"Hazard: {r['hazard']} (conf: {r['confidence']:.3f}) | Severity: {r['severity']:.4f} | Latency: {r['latency_ms']:.1f} ms")
'''
    with open(os.path.join(bundle_dir, 'inference_example.py'), 'w') as f:
        f.write(inference_script)

    readme = f"""# HazardNet Edge Deployment Bundle\n\n## Contents\n- `hazardnet_int8.tflite` - Optimized FP32 model (TFLite CONV_3D requires FP32)\n- `hazardnet_fp32.tflite` - FP32 baseline model\n- `labels.json` - {DeployConfig.NUM_HAZARDS} hazard class labels\n- `preprocessing_config.json` - Pre-computed normalization stats + band order\n- `inference_example.py` - Standalone inference with normalization & NDHWC transpose\n\n## Input Spec\n- Shape: (1, 15, 10, 64, 64) - [batch, channels, timesteps, height, width]\n- Normalization: z-score with pre-computed per-band mean/std\n- Transpose: NCDHW -> NDHWC handled automatically by inference script\n"""
    with open(os.path.join(bundle_dir, 'README.md'), 'w') as f:
        f.write(readme)

    print(f"\n  [OK] Bundle created: {bundle_dir}")
    for item in sorted(os.listdir(bundle_dir)):
        fpath = os.path.join(bundle_dir, item)
        if os.path.isfile(fpath):
            print(f"     {item}: {os.path.getsize(fpath) / 1024:.1f} KB")
        else:
            print(f"     {item}/ (directory)")
    return bundle_dir


def main():
    print("=" * 70)
    print("HAZARDNET EDGE DEPLOYMENT CONVERTER")
    print("=" * 70)

    model, norm_stats = load_model_and_stats()
    onnx_path = export_to_onnx(model)
    rep_samples = create_representative_dataset(norm_stats)
    fp32_path, int8_path = convert_to_tflite(onnx_path, rep_samples)
    agreement, sev_diff = golden_parity_test(model, norm_stats, int8_path)
    bundle_dir = create_deployment_bundle(norm_stats, int8_path, fp32_path)

    print("\n" + "=" * 70)
    print("[OK] DEPLOYMENT CONVERSION COMPLETE")
    print("=" * 70)
    print(f"  Bundle: {bundle_dir}")
    print(f"  Parity: {agreement:.1f}% hazard agreement, {sev_diff:.4f} severity MAE")
    # Carried into the manifest: these two numbers are the evidence that the artifact being
    # shipped is the model that was trained.
    return {'hazard_agreement_pct': round(float(agreement), 4),
            'severity_mae': round(float(sev_diff), 6),
            'bundle_dir': bundle_dir}


beat('converting', message='PyTorch -> ONNX -> TFLite')
CONVERSION = main()

In [ ]:
# ── Publish the run ───────────────────────────────────────────────────────────
# This cell used to print "Conversion placeholder — wire in your actual ONNX/TF steps
# above", which was misleading twice over: the converter above is 569 lines of real
# PyTorch → ONNX → TFLite with a golden parity test, and the cell after this one cloned
# the repository with a personal access token and opened a pull request from inside the
# notebook. The token is gone. What replaces both is the handshake that makes an
# unattended run auditable: a manifest naming every artifact with its size and sha256,
# the fold metrics the training actually reported, the parity numbers the converter
# measured, and the environment that produced them.
#
# `model_intake.yml` fetches this directory with `colab download`, validates the manifest
# against the bytes, and opens the pull request. Nothing here decides whether a model
# ships: the artifact enters the registry as a `candidate`, and `champion` still needs
# `python -m mlops.cli promote --by <approver>`.
import platform

try:
    import tensorflow as tf
    TF_VERSION = tf.__version__
except Exception as _exc:  # the converter has already proven it imports; be safe anyway
    TF_VERSION = f'unavailable ({_exc.__class__.__name__})'

ENVIRONMENT = {
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'cuda': torch.version.cuda,
    'torch': torch.__version__,
    'tensorflow': TF_VERSION,
    'python': platform.python_version(),
    'ram_gb': round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1),
    'resumed': bool(RESUMED_FROM),
}

# Published twice: to the VM, where `colab download` can reach it while the session
# lives, and to Drive, where it outlives the session that made it. Intake prefers the VM
# and falls back to Drive, because by the time a run finishes the VM is often already
# gone — that is what the free tier does.
PUBLISH_DIR = VM_RUN_DIR / 'bundle'
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)
for _name in rs.REQUIRED_ARTIFACTS:
    _source = Path(BUNDLE_DIR) / _name
    if not _source.is_file():
        raise SystemExit(f'the converter did not produce {_source} — refusing to publish a partial bundle')
    shutil.copy2(_source, PUBLISH_DIR / _name)

_notebook_on_vm = VM_RUN_DIR / 'HazardNet_auto_train.ipynb'
NOTEBOOK_SHA = NOTEBOOK_SHA256 or (
    rs.sha256_file(_notebook_on_vm) if _notebook_on_vm.is_file() else 'unknown (run interactively)'
)

MANIFEST = rs.manifest_doc(
    RUN_ID,
    strategy=RUN_STRATEGY,
    folds=[{key: row.get(key) for key in ('fold', 'strategy', 'accuracy', 'f1', 'rmse', 'mae', 'r2', 'n_test')}
           for row in FOLD_RESULTS],
    parity={key: CONVERSION[key] for key in ('hazard_agreement_pct', 'severity_mae')},
    environment=ENVIRONMENT,
    artifacts=rs.inventory(BUNDLE_DIR),
    notebook_sha256=NOTEBOOK_SHA,
    repo_sha=REPO_SHA or 'unknown (run interactively)',
    started_at=RUN_STARTED_STAMP,
    duration_seconds=time.time() - RUN_STARTED_AT,
    resumed_from=RESUMED_FROM,
)

# Validate before publishing, with the same function intake will use. Failing here says
# which field is wrong while the session is still alive and the log can be read; failing
# at intake says it hours later, to a pull request nobody can act on.
PROBLEMS = rs.validate_manifest(MANIFEST)
if PROBLEMS:
    raise SystemExit('the manifest this run would publish is not promotable:\n  - ' + '\n  - '.join(PROBLEMS))

rs.atomic_write_json(PUBLISH_DIR / 'run_manifest.json', MANIFEST)
rs.atomic_write_json(VM_RUN_DIR / 'run_manifest.json', MANIFEST)
if RUN_DRIVE_DIR is not None:
    # Interactive runs keep a Drive copy. Unattended ones have no Drive and do not need one:
    # the watcher carries the bundle out to a workflow artifact on the tick that sees the
    # manifest, and intake collects it from there.
    rs.atomic_write_json(RUN_DRIVE_DIR / 'run_manifest.json', MANIFEST)
    shutil.copytree(PUBLISH_DIR, RUN_DRIVE_DIR / 'bundle', dirs_exist_ok=True)

beat('published', message=f'{len(MANIFEST["artifacts"])} artifacts, {MANIFEST["duration_seconds"] / 3600:.1f} h')

print(f'[OK] RUN PUBLISHED  {RUN_ID}')
print(f'  strategy   {RUN_STRATEGY}   folds {len(MANIFEST["folds"])}')
print(f'  parity     {MANIFEST["parity"]["hazard_agreement_pct"]}% hazard agreement, '
      f'{MANIFEST["parity"]["severity_mae"]} severity MAE')
print(f'  duration   {MANIFEST["duration_seconds"] / 3600:.2f} h'
      + (f'  (resumed from {", ".join(RESUMED_FROM)})' if RESUMED_FROM else ''))
print(f'  gpu        {ENVIRONMENT["gpu"]}  torch {ENVIRONMENT["torch"]}  tf {ENVIRONMENT["tensorflow"]}')
for _entry in MANIFEST['artifacts']:
    print(f'  · {_entry["name"]:<32} {_entry["bytes"]:>10,} bytes  {_entry["sha256"][:12]}')
print(f'  manifest   {PUBLISH_DIR / "run_manifest.json"}')
print(f'  mirrored   '
      + (str(RUN_DRIVE_DIR / 'run_manifest.json') if RUN_DRIVE_DIR
         else 'nowhere — no Drive in a headless session; the watcher archives the bundle'))
print('  Intake collects this on its own schedule and opens the pull request; merging it is a human decision.')

In [ ]:
# Smoke-test the artifact exactly as Actions will: load with tflite-runtime
# (same 2 MB wheel used in the daily job) and run one inference.
import os
import numpy as np
import tensorflow as tf # Using tensorflow's tf.lite.Interpreter

TFLITE_PATH = os.path.join(BUNDLE_DIR, 'hazardnet_fp32.tflite')

# Check if the TFLite file exists before attempting to load
if not os.path.exists(TFLITE_PATH):
    print(f"Error: TFLite model not found at {TFLITE_PATH}")
    print("Please ensure the previous TFLite conversion step completed successfully.")
else:
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()
    dummy = np.random.randn(*inp['shape']).astype(inp['dtype'])
    interp.set_tensor(inp['index'], dummy)
    interp.invoke()
    for o in out:
        print(f"  output {o['name']}: shape={o['shape']} dtype={o['dtype']}")
    print(f"TFLite model OK  ({os.path.getsize(TFLITE_PATH)/1e6:.2f} MB)")
    beat('smoke-tested', message=f'{os.path.getsize(TFLITE_PATH)} bytes')